# ER-FTIR — Feature extraction, PCA score plot and LOO-optimized KDE

In [ ]:
from pathlib import Path
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.stats import gaussian_kde, chi2
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

# Folder layout: ER-FTIR/data/{Pellet,Oil_Pigments,Layer,Layer_T}/
DATA_ROOT = Path("data")
TOP_CODES = ["VC", "GC", "NL", "BZ", "BT"]
TARGET_CODE = "VC"
N_CLUSTERS = 3
WINDOW_SPEC = {
    "780-680": {"others": (780, 680), "pellet": (780, 680)},
    "620-570": {"others": (620, 570), "pellet": (620, 570)},
}
FEATURE_SPECS = [
    "780-680:width", "780-680:signed_p2p", "780-680:energy",
    "620-570:width", "620-570:signed_p2p", "620-570:energy",
]
BW_MIN, BW_MAX, N_BW = 0.40, 2.00, 161


## 1. Load and preprocess SPA spectra

In [ ]:
def read_spa(filepath):
    '''
    Input
    Read a file (string) *.spa
    ----------
    Output
    Return spectra, wavelenght (nm), titles
    '''
    with open(filepath, 'rb') as f:
        f.seek(564)
        Spectrum_Pts = np.fromfile(f, np.int32,1)[0]
        f.seek(30)
        SpectraTitles = np.fromfile(f, np.uint8,255)
        SpectraTitles = ''.join([chr(x) for x in SpectraTitles if x!=0])

        f.seek(576)

        Max_Wavenum=np.fromfile(f, np.single, 1)[0]
        Min_Wavenum=np.fromfile(f, np.single, 1)[0]
        # print(Min_Wavenum, Max_Wavenum, Spectrum_Pts)
        Wavenumbers = np.flip(np.linspace(Min_Wavenum, Max_Wavenum, Spectrum_Pts))

        f.seek(288);

        Flag=0
        while Flag != 3:
            Flag = np.fromfile(f, np.uint16, 1)

        DataPosition=np.fromfile(f,np.uint16, 1)
        f.seek(DataPosition[0])

        Spectra = np.fromfile(f, np.single, Spectrum_Pts)
    return Spectra, Wavenumbers, SpectraTitles

#standard normal variate pre-processing
def snv(X):
    mu = X.mean(axis=1, keepdims=True)
    sigma = X.std(axis=1, keepdims=True)
    sigma[sigma == 0] = 1.0
    return (X - mu) / sigma

In [ ]:
def load_group(folder):
    files = sorted(Path(folder).glob("*.SPA")) + sorted(Path(folder).glob("*.spa"))
    files = list(dict.fromkeys(files))
    if not files:
        raise FileNotFoundError(f"No SPA files found in {folder}")
    spectra, titles, axis = [], [], None
    for path in files:
        y, x, _ = read_spa(str(path))
        if axis is None:
            axis = np.asarray(x, dtype=float)
        elif len(x) != len(axis) or not np.allclose(x, axis):
            raise ValueError(f"Incompatible spectral axis: {path}")
        spectra.append(np.asarray(y, dtype=float))
        titles.append(path.name)
    return np.stack(spectra), titles, axis

#smoothing of the spectra
def preprocess_group(X, wn):
    X = np.asarray(X, dtype=float).copy()
    for lo, hi in [(3100, 4000), (1300, 2000), (400, 900)]:
        mask = (wn >= lo) & (wn <= hi)
        if mask.sum() < 201:
            raise ValueError(f"Not enough points in {lo}–{hi} cm^-1")
        X[:, mask] = savgol_filter(X[:, mask], 201, 3, axis=1)
    X = snv(X)
    delta = float(np.mean(np.diff(wn)))
    return savgol_filter(X, 201, 3, deriv=1, delta=delta, axis=1)
    
#sample loading: pellet, oil on glass, raw and treated canvas samples (or whatever it is needed)
pellet, local_title, wavelength_tmp = load_group(DATA_ROOT / "Pellet")
oil, title_o, wn_o = load_group(DATA_ROOT / "Oil_Pigments")
raw, title_R, wn_r = load_group(DATA_ROOT / "Layer")
treated, title_T, wn_t = load_group(DATA_ROOT / "Layer_T")
for axis in (wn_o, wn_r, wn_t):
    if len(axis) != len(wavelength_tmp) or not np.allclose(axis, wavelength_tmp):
        raise ValueError("All four groups must share the same wavenumber grid")

X_deriv1_p = preprocess_group(pellet, wavelength_tmp)
X_deriv5 = preprocess_group(oil, wavelength_tmp)
X_deriv1 = preprocess_group(raw, wavelength_tmp)
X_deriv4 = preprocess_group(treated, wavelength_tmp)

## 2. Construct the window-based features

In [ ]:
def clean_name(s):
    s = str(s).strip()
    if s.lower().endswith(".spa"):
        s = s[:-4]
    return s

def strip_terminal_T(name):
    s = clean_name(name)
    if s.endswith("_T"):
        s = s[:-2]
    return s

def parse_ratio_token(tok):
    t = str(tok).strip()

    if t == "05":
        return 0.5

    try:
        return float(t)
    except ValueError:
        pass

    if t.isdigit():
        return float(int(t))

    raise ValueError(f"Token diluizione non riconosciuto: {tok}")

def format_ratio_for_label(r):
    if r is None:
        return "-"
    if np.isclose(r, round(r)):
        return str(int(round(r)))
    return str(r)

def normalize_top_codes(top_codes):
    if isinstance(top_codes, str):
        return [top_codes]
    return list(top_codes)

def parse_two_layer_name(name):
    """
    One to two paint layers expected over the support
    Name format accepted:
    
    1) Two layers
       X_N_Y_M
       X_N_Y_M_T   _T means on treated canvas

       Y_M -> top layer code and binder-to-pigment ratio
       X_N -> lower layer code and binder-to-pigment ratio

    2) single paint layer
       X_N
       X_N_T

    3) Multi-layer of the same pigment
       X_N_X_N
       X_N_X_N_T
       X_N_X_N_X_N
       X_N_X_N_X_N_T
    """
    s = strip_terminal_T(name)
    parts = s.split("_")

    out = {
        "display_name": s,
        "stack_signature": s,
    }

    # singel paint layer case: X_N
    if len(parts) == 2:
        x, n = parts
        out.update({
            "bottom_code": None,
            "bottom_ratio": None,
            "top_code": x,
            "top_ratio": parse_ratio_token(n),
            "stack_type": "single_layer",
            "n_repeated_layers": 1,
        })
        return out

    # Double paint layer case: X_N_Y_M
    if len(parts) == 4:
        x, n, y, m = parts

        # Double paint layer same pigment case: X_N_X_N
        if x == y and n == m:
            out.update({
                "bottom_code": None,
                "bottom_ratio": None,
                "top_code": x,
                "top_ratio": parse_ratio_token(n),
                "stack_type": "repeated_same_pigment",
                "n_repeated_layers": 2,
            })
            return out

        out.update({
            "bottom_code": x,
            "bottom_ratio": parse_ratio_token(n),
            "top_code": y,
            "top_ratio": parse_ratio_token(m),
            "stack_type": "two_layer",
            "n_repeated_layers": 1,
        })
        return out

    # more than two paint layers of the same pigment (treated as a double layer of the same pigment): X_N_X_N_X_N ...
    if len(parts) % 2 == 0 and len(parts) >= 6:
        codes = parts[::2]
        ratios = parts[1::2]

        if len(set(codes)) == 1 and len(set(ratios)) == 1:
            out.update({
                "bottom_code": None,
                "bottom_ratio": None,
                "top_code": codes[0],
                "top_ratio": parse_ratio_token(ratios[0]),
                "stack_type": "repeated_same_pigment",
                "n_repeated_layers": len(codes),
            })
            return out

    raise ValueError(f"Nome non nel formato supportato: {name}")

def get_oil_title_from_code_ratio(code, ratio):
    if code not in CODE_TO_OIL_NAME:
        raise KeyError(f"Codice pigmento non mappato: {code}")
    return f"{CODE_TO_OIL_NAME[code]}_oil_{format_ratio_for_label(ratio)}"

def get_pellet_title_from_code(code):
    if code not in CODE_TO_PELLET_NAME:
        raise KeyError(f"Codice pellet non mappato: {code}")
    return CODE_TO_PELLET_NAME[code]

def find_exact_title_index(title_list, target_title_no_ext):
    for i, t in enumerate(title_list):
        if clean_name(t) == target_title_no_ext:
            return i
    raise KeyError(f"Titolo non trovato: {target_title_no_ext}")

def prepare_common_axis(wavelength_tmp, xlim=(4000, 400)):
    wn = np.asarray(wavelength_tmp, dtype=float)
    lo, hi = min(xlim), max(xlim)
    mask = (wn >= lo) & (wn <= hi)
    wn_plot = wn[mask]

    reverse_needed = wn_plot[0] > wn_plot[-1]
    if reverse_needed:
        wn_plot = wn_plot[::-1]

    return wn_plot, mask, reverse_needed

def get_masked_row(X, idx, mask, reverse_needed):
    y = np.asarray(X[idx], dtype=float)[mask]
    if reverse_needed:
        y = y[::-1]
    return y

# Morphologiacl features construction
def compute_window_features(wn, y):
    """
    signed p2p:
       excursion between the derivative extrema, whereas its sign is determined by their relative ordering along
       the increasing wavenumber axis
    width:
      |wmax - wmin|
    energy:
      integral(y^2 dwn)
    w0:
      zero crossing between maximum and minimum on the mean-centered signal
    w0_no_norm:
      zero crossing between maximum and minimum on the raw signal
    p2p_over_width:
      signed_p2p / width

    All quantities are computed on the raw signals, while only w0 is computed on the mean-centered signal
    """
    wn = np.asarray(wn, dtype=float)
    y = np.asarray(y, dtype=float)

    # -----------------------------
    # quantità RAW
    # -----------------------------
    i_max = int(np.argmax(y))
    i_min = int(np.argmin(y))

    ymax = y[i_max]
    ymin = y[i_min]
    wmax = wn[i_max]
    wmin = wn[i_min]

    p2p = ymax - ymin
    signed_p2p = p2p if i_max < i_min else -p2p
    width = abs(wmax - wmin)
    energy = np.trapz(y**2, wn)

    # -----------------------------
    # w0 sul RAW
    # -----------------------------
    i1, i2 = sorted([i_min, i_max])
    w0_no_norm = np.nan

    for i in range(i1, i2):
        y1, y2 = y[i], y[i + 1]
        if y1 == 0:
            w0_no_norm = wn[i]
            break
        if y1 * y2 < 0:
            x1, x2 = wn[i], wn[i + 1]
            w0_no_norm = x1 - y1 * (x2 - x1) / (y2 - y1)
            break

    # -----------------------------
    # w0 sul mean-centered
    # ma tra lo stesso min/max del RAW
    # -----------------------------
    y_mc = mean_center_signal(y)
    w0 = np.nan

    for i in range(i1, i2):
        y1, y2 = y_mc[i], y_mc[i + 1]
        if y1 == 0:
            w0 = wn[i]
            break
        if y1 * y2 < 0:
            x1, x2 = wn[i], wn[i + 1]
            w0 = x1 - y1 * (x2 - x1) / (y2 - y1)
            break

    p2p_over_width = signed_p2p / width if width > 0 else np.nan

    return {
        "signed_p2p": signed_p2p,
        "width": width,
        "energy": energy,
        "ymin": ymin,
        "ymax": ymax,
        "wmin": wmin,
        "wmax": wmax,
        "w0": w0,                  # questo diventa quello mean-centered
        "w0_no_norm": w0_no_norm,  # questo è il crossing raw
        "p2p_over_width": p2p_over_width,
    }

def build_layer_label(info, support_tag):
   
    if info["bottom_code"] is None:
        return f"{info['display_name']} [{support_tag}]"
    return (
        f"{info['bottom_code']}_{format_ratio_for_label(info['bottom_ratio'])}_"
        f"{info['top_code']}_{format_ratio_for_label(info['top_ratio'])} [{support_tag}]"
    )

def normalize_window_spec(window_spec):
    """
    Accept:
      - {"780-680": (780,680), ...}
      - {"780-680": {"others": (780,680), "pellet": (780,680)}, ...}

    gives in output:
      {
        "780-680": {
            "others": (780,680),
            "pellet": (780,680)
        },
        ...
      }
    """
    out = {}
    for wname, val in window_spec.items():
        if isinstance(val, dict):
            if "others" not in val:
                raise ValueError(f"Per la finestra {wname} manca 'others'")
            others = tuple(val["others"])
            pellet = tuple(val.get("pellet", val["others"]))
        else:
            others = tuple(val)
            pellet = tuple(val)
        out[wname] = {"others": others, "pellet": pellet}
    return out

def union_bounds(bounds_a, bounds_b):
    
    a1, a2 = bounds_a
    b1, b2 = bounds_b
    return (max(a1, a2, b1, b2), min(a1, a2, b1, b2))

def collect_top_code_samples_with_refs(
    top_code,
    X_deriv1,
    title_R,
    X_deriv4,
    title_T,
    X_deriv5,
    title_o,
    X_deriv1_p,
    local_title,
    wavelength_tmp,
    *,
    xlim=(4000, 400),
):
    wn_plot, mask, reverse_needed = prepare_common_axis(wavelength_tmp, xlim=xlim)

    X_raw = np.asarray(X_deriv1, dtype=float)
    X_treated = np.asarray(X_deriv4, dtype=float)
    X_oil = np.asarray(X_deriv5, dtype=float)
    X_pellet = np.asarray(X_deriv1_p, dtype=float)

    samples = []

    # layered raw
    for i, t in enumerate(title_R):
        info = parse_two_layer_name(t)
        if info["top_code"] != top_code:
            continue

        y = get_masked_row(X_raw, i, mask, reverse_needed)
        label = build_layer_label(info, "R")

        samples.append({
            "kind": "layered_raw",
            "support": "R",
            "sample_name": strip_terminal_T(t),
            "display_name": info["display_name"],
            "stack_signature": info["stack_signature"],
            "stack_type": info["stack_type"],
            "n_repeated_layers": info["n_repeated_layers"],
            "top_code": info["top_code"],
            "top_ratio": info["top_ratio"],
            "bottom_code": info["bottom_code"],
            "bottom_ratio": info["bottom_ratio"],
            "label": label,
            "y": y,
        })

    # layered treated
    for i, t in enumerate(title_T):
        info = parse_two_layer_name(t)
        if info["top_code"] != top_code:
            continue

        y = get_masked_row(X_treated, i, mask, reverse_needed)
        label = build_layer_label(info, "T")

        samples.append({
            "kind": "layered_treated",
            "support": "T",
            "sample_name": strip_terminal_T(t),
            "display_name": info["display_name"],
            "stack_signature": info["stack_signature"],
            "stack_type": info["stack_type"],
            "n_repeated_layers": info["n_repeated_layers"],
            "top_code": info["top_code"],
            "top_ratio": info["top_ratio"],
            "bottom_code": info["bottom_code"],
            "bottom_ratio": info["bottom_ratio"],
            "label": label,
            "y": y,
        })

    if len(samples) == 0:
        raise ValueError(f"No sample found for top code = {top_code}")

    # diluizioni top presenti
    top_ratios_present = sorted(set(s["top_ratio"] for s in samples if s["top_ratio"] is not None))

    # refs oil per ogni diluizione presente
    for ratio in top_ratios_present:
        try:
            oil_title = get_oil_title_from_code_ratio(top_code, ratio)
            idx_o = find_exact_title_index(title_o, oil_title)
            y = get_masked_row(X_oil, idx_o, mask, reverse_needed)

            samples.append({
                "kind": "oil_ref",
                "support": "oil",
                "sample_name": oil_title,
                "display_name": oil_title,
                "stack_signature": oil_title,
                "stack_type": "oil_ref",
                "n_repeated_layers": None,
                "top_code": top_code,
                "top_ratio": ratio,
                "bottom_code": None,
                "bottom_ratio": None,
                "label": f"{top_code}_{format_ratio_for_label(ratio)} oil ref",
                "y": y,
            })

        except KeyError:
            print(f"[INFO] Oil ref not found for {top_code} ratio={ratio}, skip.")
            continue

    # pellet unico
    pellet_title = get_pellet_title_from_code(top_code)
    idx_p = find_exact_title_index(local_title, pellet_title)
    y_p = get_masked_row(X_pellet, idx_p, mask, reverse_needed)

    samples.append({
        "kind": "pellet_ref",
        "support": "pellet",
        "sample_name": pellet_title,
        "display_name": pellet_title,
        "stack_signature": pellet_title,
        "stack_type": "pellet_ref",
        "n_repeated_layers": None,
        "top_code": top_code,
        "top_ratio": None,
        "bottom_code": None,
        "bottom_ratio": None,
        "label": f"{top_code} pellet",
        "y": y_p,
    })

    return wn_plot, samples, top_ratios_present

def compute_features_for_top_code_flexible(
    top_code,
    X_deriv1,
    title_R,
    X_deriv4,
    title_T,
    X_deriv5,
    title_o,
    X_deriv1_p,
    local_title,
    wavelength_tmp,
    *,
    window_spec,
    xlim=(4000, 400),
):
    window_spec = normalize_window_spec(window_spec)

    wn_plot, samples, top_ratios_present = collect_top_code_samples_with_refs(
        top_code=top_code,
        X_deriv1=X_deriv1,
        title_R=title_R,
        X_deriv4=X_deriv4,
        title_T=title_T,
        X_deriv5=X_deriv5,
        title_o=title_o,
        X_deriv1_p=X_deriv1_p,
        local_title=local_title,
        wavelength_tmp=wavelength_tmp,
        xlim=xlim,
    )

    rows = []

    for s in samples:
        is_pellet = (s["kind"] == "pellet_ref")

        for wname, wb in window_spec.items():
            bounds = wb["pellet"] if is_pellet else wb["others"]

            lo, hi = min(bounds), max(bounds)
            m = (wn_plot >= lo) & (wn_plot <= hi)

            if np.sum(m) < 3:
                continue

            feat = compute_window_features(wn_plot[m], s["y"][m])

            row = {
                "window": wname,
                "window_used": tuple(bounds),
                "label": s["label"],
                "display_name": s["display_name"],
                "stack_signature": s["stack_signature"],
                "stack_type": s["stack_type"],
                "n_repeated_layers": s["n_repeated_layers"],
                "kind": s["kind"],
                "support": s["support"],
                "top_code": s["top_code"],
                "top_ratio": s["top_ratio"],
                "bottom_code": s["bottom_code"],
                "bottom_ratio": s["bottom_ratio"],
            }
            row.update(feat)
            rows.append(row)

    return wn_plot, samples, rows, top_ratios_present

def compute_features_for_multiple_top_codes_flexible(
    top_codes,
    X_deriv1,
    title_R,
    X_deriv4,
    title_T,
    X_deriv5,
    title_o,
    X_deriv1_p,
    local_title,
    wavelength_tmp,
    *,
    window_spec,
    xlim=(4000, 400),
):
    top_codes = normalize_top_codes(top_codes)

    all_samples = []
    all_rows = []
    wn_plot_ref = None
    ratios_by_top = {}

    for tc in top_codes:
        wn_plot, samples, rows, top_ratios_present = compute_features_for_top_code_flexible(
            top_code=tc,
            X_deriv1=X_deriv1,
            title_R=title_R,
            X_deriv4=X_deriv4,
            title_T=title_T,
            X_deriv5=X_deriv5,
            title_o=title_o,
            X_deriv1_p=X_deriv1_p,
            local_title=local_title,
            wavelength_tmp=wavelength_tmp,
            window_spec=window_spec,
            xlim=xlim,
        )

        if wn_plot_ref is None:
            wn_plot_ref = wn_plot

        all_samples.extend(samples)
        all_rows.extend(rows)
        ratios_by_top[tc] = top_ratios_present

    return wn_plot_ref, all_samples, all_rows, ratios_by_top

def build_feature_dataframe(rows_top):
    df = pd.DataFrame(rows_top).copy()

    if df.empty:
        raise ValueError("rows_top è vuoto")

    pellet_df = (
        df[df["kind"] == "pellet_ref"]
        [["window", "top_code", "w0", "w0_no_norm", "width", "p2p_over_width", "energy"]]
        .rename(columns={
            "w0": "pellet_w0",
            "w0_no_norm": "pellet_w0_no_norm",
            "width": "pellet_width",
            "p2p_over_width": "pellet_p2pw",
            "energy": "pellet_energy",
        })
    )

    df = df.merge(pellet_df, on=["window", "top_code"], how="left")

    df["delta_w0_vs_pellet"] = df["w0"] - df["pellet_w0"]
    df["delta_width_vs_pellet"] = df["width"] - df["pellet_width"]
    df["ratio_p2pw_vs_pellet"] = df["p2p_over_width"] / df["pellet_p2pw"]
    df["ratio_energy_vs_pellet"] = df["energy"] / df["pellet_energy"]

    return df

def make_clean_feature_table(
    df,
    window_name,
    include_single_layer=False,
    top_codes=None,
    round_digits=4
):
    sub = df[df["window"] == window_name].copy()

    if top_codes is not None:
        top_codes = normalize_top_codes(top_codes)
        sub = sub[sub["top_code"].isin(top_codes)].copy()

    if not include_single_layer:
        sub = sub[
            ~(
                (sub["kind"].isin(["layered_raw", "layered_treated"]))
                & (sub["bottom_code"].isna())
            )
        ].copy()

    out_parts = []

    for tc, grp in sub.groupby("top_code"):
        pel = grp[grp["kind"] == "pellet_ref"]
        if len(pel) == 0:
            continue
        pel = pel.iloc[0]

        w0_pellet = pel["w0"]
        width_pellet = pel["width"]
        sharp_pellet = pel["p2p_over_width"]
        energy_pellet = pel["energy"]

        g = grp.copy()
        g["dw0_rel"] = (g["w0"] - w0_pellet) / w0_pellet
        g["dwid_rel"] = (g["width"] - width_pellet) / width_pellet
        g["sharp_rel"] = g["p2p_over_width"] / sharp_pellet
        g["E_rel"] = g["energy"] / energy_pellet
        out_parts.append(g)

    if len(out_parts) == 0:
        return pd.DataFrame()

    sub = pd.concat(out_parts, ignore_index=True)

    table = pd.DataFrame({
        "top": sub["top_code"],
        "sample": sub["label"],
        "type": sub["kind"].map({
            "layered_raw": "R",
            "layered_treated": "T",
            "oil_ref": "oil",
            "pellet_ref": "pel"
        }),
        "bot": sub["bottom_code"].fillna("-"),
        "r": sub["top_ratio"].fillna("-"),
        "wmin": sub["wmin"],
        "wmax": sub["wmax"],
        "w0": sub["w0"],                       # mean-centered
        "w0 (no norm)": sub["w0_no_norm"],    # raw
        "wid": sub["width"],
        "sharp": sub["p2p_over_width"],
        "E": sub["energy"],
        "dw0_rel": sub["dw0_rel"],
        "dwid_rel": sub["dwid_rel"],
        "sharp_rel": sub["sharp_rel"],
        "E_rel": sub["E_rel"],
    })

    table = table.sort_values(by=["top", "w0"])

    for col in ["wmin", "wmax", "w0", "w0 (no norm)", "wid", "sharp", "E", "dw0_rel", "dwid_rel", "sharp_rel", "E_rel"]:
        table[col] = table[col].astype(float).round(round_digits)

    return table

def print_clean_table(
    df,
    window_name,
    include_single_layer=False,
    top_codes=None
):
    table = make_clean_feature_table(
        df,
        window_name,
        include_single_layer=include_single_layer,
        top_codes=top_codes
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.width", 260
    ):
        print(f"\n===== CLEAN TABLE | {window_name} =====")
        print(table)

def plot_sample_spectra_flexible(
    wn_plot,
    samples,
    window_spec,
    *,
    top_code=None,
    mode="separate",
    normalize=False,
    figsize_per_window=(8, 5),
    spectrum_lw=1.2,
    alpha_layered=0.9,
    alpha_ref=1.0,
    show_legend=True,
):
    window_spec = normalize_window_spec(window_spec)

    def _local_normalize_l2(y):
        y = np.asarray(y, dtype=float)
        y0 = y - np.mean(y)
        nrm = np.linalg.norm(y0)
        if np.isclose(nrm, 0):
            return y0
        return y0 #/ nrm

    kind_style = {
        "layered_raw": dict(color="tab:blue", linestyle="-", alpha=alpha_layered),
        "layered_treated": dict(color="tab:orange", linestyle="-", alpha=alpha_layered),
        "oil_ref": dict(color="tab:green", linestyle="--", alpha=alpha_ref),
        "pellet_ref": dict(color="tab:red", linestyle=":", alpha=alpha_ref),
    }

    kind_legend = {
        "layered_raw": "layered R",
        "layered_treated": "layered T",
        "oil_ref": "oil ref",
        "pellet_ref": "pellet",
    }

    window_union = {
        wname: union_bounds(wb["others"], wb["pellet"])
        for wname, wb in window_spec.items()
    }

    if mode == "separate":
        for wname, bounds in window_union.items():
            lo, hi = min(bounds), max(bounds)
            m = (wn_plot >= lo) & (wn_plot <= hi)

            if np.sum(m) < 3:
                print(f"Finestra {wname} troppo piccola o vuota, salto.")
                continue

            x = wn_plot[m]
            fig, ax = plt.subplots(figsize=figsize_per_window)
            used_kind_label = set()

            for s in samples:
                y = s["y"][m]
                if normalize:
                    y = _local_normalize_l2(y)

                style = kind_style[s["kind"]].copy()

                label = None
                if s["kind"] not in used_kind_label:
                    label = kind_legend[s["kind"]]
                    used_kind_label.add(s["kind"])

                ax.plot(x, y, lw=spectrum_lw, label=label, **style)

                ax.text(
                    x[len(x) // 2],
                    y[len(y) // 2],
                    " " + s["label"],
                    fontsize=7,
                    alpha=0.85,
                )

            ax.axhline(0, color="black", lw=0.7)
            ax.set_xlim(x[0], x[-1])
            ax.invert_xaxis()
            ax.grid(True, alpha=0.25)
            ax.set_xlabel("Wavenumber (cm$^{-1}$)")
            ax.set_ylabel("Normalized signal" if normalize else "Derivative signal")

            ttl = f"{wname}"
            if top_code is not None:
                ttl = f"{top_code} | {wname}"
            if normalize:
                ttl += " | normalized"
            ax.set_title(ttl)

            if show_legend:
                ax.legend(fontsize=8)

            plt.tight_layout()
            plt.show()

    elif mode == "grid":
        nwin = len(window_union)
        fig, axes = plt.subplots(
            1, nwin,
            figsize=(figsize_per_window[0] * nwin, figsize_per_window[1]),
            squeeze=False
        )
        axes = axes.ravel()

        for ax, (wname, bounds) in zip(axes, window_union.items()):
            lo, hi = min(bounds), max(bounds)
            m = (wn_plot >= lo) & (wn_plot <= hi)

            if np.sum(m) < 3:
                ax.set_visible(False)
                continue

            x = wn_plot[m]
            used_kind_label = set()

            for s in samples:
                y = s["y"][m]
                if normalize:
                    y = _local_normalize_l2(y)

                style = kind_style[s["kind"]].copy()

                label = None
                if s["kind"] not in used_kind_label:
                    label = kind_legend[s["kind"]]
                    used_kind_label.add(s["kind"])

                ax.plot(x, y, lw=spectrum_lw, label=label, **style)

            ax.axhline(0, color="black", lw=0.7)
            ax.set_xlim(x[0], x[-1])
            ax.invert_xaxis()
            ax.grid(True, alpha=0.25)
            ax.set_xlabel("Wavenumber (cm$^{-1}$)")
            ax.set_title(wname + (" | normalized" if normalize else ""))

        axes[0].set_ylabel("Normalized signal" if normalize else "Derivative signal")

        if show_legend:
            axes[0].legend(fontsize=8)

        title = "Sample spectra in selected windows"
        if top_code is not None:
            title = f"{top_code} | sample spectra in selected windows"
        fig.suptitle(title, fontsize=13)

        plt.tight_layout()
        plt.show()

    else:
        raise ValueError("mode deve essere 'separate' oppure 'grid'")

def run_pigment_feature_analysis(
    top_codes,
    X_deriv1,
    title_R,
    X_deriv4,
    title_T,
    X_deriv5,
    title_o,
    X_deriv1_p,
    local_title,
    wavelength_tmp,
    *,
    window_spec,
    xlim=(4000, 400),
    include_single_layer=True,
    annotate_biplot=True,
    annotate_delta=True,
    plot_spectra=True,
    spectra_mode="separate",
    spectra_normalize=False,
):
    """
    Complete pipeline for several top codes.

    window_spec can be:
      1) {
           "780-680": (780, 680),
           "620-570": (620, 570),
         }

      2) {
           "780-680": {"others": (780,680), "pellet": (790,690)},
           "620-570": {"others": (620,570), "pellet": (625,575)},
         }
    """
    top_codes = normalize_top_codes(top_codes)

    wn_plot, samples, rows, ratios_by_top = compute_features_for_multiple_top_codes_flexible(
        top_codes=top_codes,
        X_deriv1=X_deriv1,
        title_R=title_R,
        X_deriv4=X_deriv4,
        title_T=title_T,
        X_deriv5=X_deriv5,
        title_o=title_o,
        X_deriv1_p=X_deriv1_p,
        local_title=local_title,
        wavelength_tmp=wavelength_tmp,
        window_spec=window_spec,
        xlim=xlim,
    )

    df_feat = build_feature_dataframe(rows)

    print(f"\nTOP CODES = {top_codes}")
    for tc, rr in ratios_by_top.items():
        print(f"{tc}: diluizioni top presenti = {[format_ratio_for_label(r) for r in rr]}")

    for wname in normalize_window_spec(window_spec).keys():
        print_clean_table(
            df_feat,
            wname,
            include_single_layer=include_single_layer,
            top_codes=top_codes
        )
    
    if plot_spectra:
        plot_sample_spectra_flexible(
            wn_plot,
            samples,
            window_spec,
            top_code=",".join(top_codes),
            mode=spectra_mode,
            normalize=spectra_normalize
        )

    return {
        "wn_plot": wn_plot,
        "samples": samples,
        "rows": rows,
        "df_feat": df_feat,
        "ratios_by_top": ratios_by_top,
    }

In [ ]:
out = run_pigment_feature_analysis(
    top_codes=TOP_CODES,
    X_deriv1=X_deriv1, title_R=title_R,
    X_deriv4=X_deriv4, title_T=title_T,
    X_deriv5=X_deriv5, title_o=title_o,
    X_deriv1_p=X_deriv1_p, local_title=local_title,
    wavelength_tmp=wavelength_tmp,
    window_spec=WINDOW_SPEC,
    plot_spectra=False,
)


## 3. PCA score plot and HCA

In [ ]:
def _get_feature_source_df_from_out(out):
    if "rows" not in out:
        raise ValueError("out non contiene 'rows'")
    return pd.DataFrame(out["rows"]).copy()

def _normalize_feature_specs(feature_specs):
    """
    Accetta:
      - ["970-900:energy", "620-570:w0"]
      - [("970-900","energy"), ("620-570","w0")]
    Ritorna lista di tuple [(window, feature), ...]
    """
    out_specs = []
    for fs in feature_specs:
        if isinstance(fs, str):
            if ":" not in fs:
                raise ValueError(f"Invalid feature spec: {fs}. Use 'window:feature'")
            w, f = fs.split(":", 1)
            out_specs.append((w.strip(), f.strip()))
        elif isinstance(fs, (tuple, list)) and len(fs) == 2:
            out_specs.append((str(fs[0]).strip(), str(fs[1]).strip()))
        else:
            raise ValueError(f"Invalid feature spec: {fs}")
    return out_specs

def _make_sample_id(row):
    return f"{row['top_code']}:{row['label']}:{row['support']}"

def _kind_marker(kind):
    return {
        "layered_raw": "s",
        "layered_treated": "o",
        "oil_ref": "^",
        "pellet_ref": "D",
    }.get(kind, "x")

def _build_metadata_per_sample(df):
    meta_cols = [
        "label", "top_code", "kind", "support",
        "display_name", "stack_signature", "stack_type",
        "n_repeated_layers", "top_ratio", "bottom_code", "bottom_ratio"
    ]
    meta_cols = [c for c in meta_cols if c in df.columns]

    meta = (
        df[meta_cols]
        .drop_duplicates(subset=["top_code", "label"])
        .copy()
    )
    meta["sample_id"] = meta.apply(_make_sample_id, axis=1)
    meta = meta.set_index("sample_id")
    return meta

def build_feature_vector_table_from_out(
    out,
    feature_specs,
    *,
    top_codes=None,
    include_kinds=("layered_raw", "layered_treated", "oil_ref", "pellet_ref"),
    include_single_layer=True,
    dropna="any",   # "any" oppure "none"
):
    """
    Construct a table: one sample per row, selected feature per column
    """
    df = _get_feature_source_df_from_out(out)
    specs = _normalize_feature_specs(feature_specs)

    if top_codes is not None:
        if isinstance(top_codes, str):
            top_codes = [top_codes]
        df = df[df["top_code"].isin(top_codes)].copy()

    if include_kinds is not None:
        df = df[df["kind"].isin(include_kinds)].copy()

    if not include_single_layer:
        df = df[
            ~(
                (df["kind"].isin(["layered_raw", "layered_treated"]))
                & (df["bottom_code"].isna())
            )
        ].copy()

    if df.empty:
        raise ValueError("Nessun dato disponibile dopo i filtri")

    meta = _build_metadata_per_sample(df)

    rows = []
    sample_ids = []

    df = df.copy()
    df["sample_id"] = df.apply(_make_sample_id, axis=1)

    for sample_id, g in df.groupby("sample_id"):
        feat_row = {}
        ok = True

        for window_name, feat_name in specs:
            sub = g[g["window"] == window_name]
            col_name = f"{window_name}__{feat_name}"

            if len(sub) == 0:
                feat_row[col_name] = np.nan
                if dropna == "any":
                    ok = False
                continue

            if feat_name not in sub.columns:
                val = np.nan
            else:
                val = sub.iloc[0][feat_name]
            feat_row[col_name] = val

            if pd.isna(val) and dropna == "any":
                ok = False

        if ok or dropna == "none":
            rows.append(feat_row)
            sample_ids.append(sample_id)

    if len(rows) == 0:
        raise ValueError("Nessun sample ha tutte le feature richieste")

    df_vectors = pd.DataFrame(rows, index=sample_ids)
    df_vectors.index.name = "sample_id"
    df_vectors = df_vectors.reset_index()

    df_vectors = df_vectors.merge(
        meta.reset_index(),
        on="sample_id",
        how="left"
    )

    feature_cols = [f"{w}__{f}" for w, f in specs]

    ordered_cols = (
        ["sample_id", "top_code", "label", "kind", "support",
         "bottom_code", "top_ratio", "stack_type", "n_repeated_layers"]
        + feature_cols
    )
    ordered_cols = [c for c in ordered_cols if c in df_vectors.columns]
    rest = [c for c in df_vectors.columns if c not in ordered_cols]
    df_vectors = df_vectors[ordered_cols + rest]

    return df_vectors, feature_cols

def analyze_feature_vectors_from_out(
    out,
    feature_specs,
    *,
    top_codes=None,
    include_kinds=("layered_raw", "layered_treated", "oil_ref", "pellet_ref"),
    include_single_layer=True,
    dropna="any",
    scale_features=True,
    n_pcs=2,
    hca_metric="euclidean",
    hca_linkage="ward",
    clustering_space="raw",   # "raw" oppure "pca"
    n_clusters=2,
    plot_labels=True,
    figsize_dendro=(11, 5),
    figsize_pca=(8, 6),
    figsize_loadings=(9, 4),
):
    """
    clustering_space:
      - "raw": clustering on the original vectors (or standardized)
      - "pca": clustering on the first n_pcs PC 
    """
    df_vectors, feature_cols = build_feature_vector_table_from_out(
        out,
        feature_specs,
        top_codes=top_codes,
        include_kinds=include_kinds,
        include_single_layer=include_single_layer,
        dropna=dropna,
    )

    X = df_vectors[feature_cols].to_numpy(dtype=float)

    if np.isnan(X).any():
        raise ValueError("NaN encountered in vectors. Use dropna='any' or select other features")

    # -----------------------------
    # scaling opzionale
    # -----------------------------
    scaler = None
    if scale_features:
        scaler = StandardScaler()
        X_proc = scaler.fit_transform(X)
    else:
        X_proc = X.copy()

    # -----------------------------
    # PCA
    # -----------------------------
    max_pcs = min(X_proc.shape[0], X_proc.shape[1])
    n_pcs_eff = min(max(1, n_pcs), max_pcs)

    pca = PCA(n_components=n_pcs_eff)
    X_pca = pca.fit_transform(X_proc)

    # -----------------------------
    # Spazio su cui fare clustering
    # -----------------------------
    if clustering_space == "raw":
        X_cluster = X_proc
    elif clustering_space == "pca":
        X_cluster = X_pca
    else:
        raise ValueError("clustering_space must be 'raw' or 'pca'")

    # -----------------------------
    # HCA
    # -----------------------------
    Z = linkage(X_cluster, method=hca_linkage, metric=hca_metric)

    # -----------------------------
    # Clustering flat
    # -----------------------------
    if n_clusters is not None and n_clusters >= 2:
        try:
            cluster_model = AgglomerativeClustering(
                n_clusters=n_clusters,
                metric="euclidean",
                linkage="ward" if hca_linkage == "ward" else "average"
            )
            cluster_labels = cluster_model.fit_predict(X_cluster)
        except Exception:
            cluster_labels = fcluster(Z, t=n_clusters, criterion="maxclust") - 1
    else:
        cluster_labels = np.zeros(len(df_vectors), dtype=int)

    df_vectors["cluster"] = cluster_labels

        # -----------------------------
    # Colori per top_code
    # -----------------------------
    top_codes_unique = sorted(df_vectors["top_code"].dropna().unique().tolist())
    cmap = plt.get_cmap("tab20")
    color_map = {tc: cmap(i % 20) for i, tc in enumerate(top_codes_unique)}

    # -----------------------------
    # DENDROGRAMMA
    # -----------------------------
    plt.figure(figsize=figsize_dendro)
    dendrogram(
        Z,
        labels=df_vectors["sample_id"].tolist(),
        leaf_rotation=90
    )
    plt.title(f"Hierarchical clustering ({clustering_space} space)")
    plt.ylabel("linkage distance")
    plt.tight_layout()
    plt.show()


    # -----------------------------
    # PCA scatter + loadings
    # -----------------------------
    if n_pcs_eff >= 2:
        fig = plt.figure(figsize=(12, 8))

        gs = fig.add_gridspec(
            2, 2,
            width_ratios=[5.8, 1.8],
            height_ratios=[1.6, 5.8],
            wspace=0.16,
            hspace=0.18
        )

        ax_top = fig.add_subplot(gs[0, 0])
        ax_sc = fig.add_subplot(gs[1, 0])
        ax_right = fig.add_subplot(gs[1, 1])

        # -------------------------
        # PCA scatter
        # -------------------------
        for _, row in df_vectors.iterrows():
            idx = row.name
            ax_sc.scatter(
                X_pca[idx, 0],
                X_pca[idx, 1],
                s=100,
                marker=_kind_marker(row["kind"]),
                color=color_map.get(row["top_code"], "gray"),
                alpha=0.9
            )

            if plot_labels:
                ax_sc.text(
                    X_pca[idx, 0],
                    X_pca[idx, 1],
                    " " + row["sample_id"],
                    fontsize=7
                )

        handles = [
            plt.Line2D([0], [0],
                      marker="o", linestyle="",
                      markerfacecolor=color_map[tc],
                      markeredgecolor=color_map[tc],
                      markersize=7, label=tc)
            for tc in top_codes_unique
        ]
        ax_sc.legend(
            handles=handles,
            title="top_code",
            loc="upper right",
            fontsize=7,
            title_fontsize=8,
            frameon=True,
            borderpad=0.3,
            handletextpad=0.4,
            labelspacing=0.3
        )

        ax_sc.set_xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.1f}%)")
        ax_sc.set_ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.1f}%)")
        ax_sc.set_title("PCA of selected feature vectors")
        ax_sc.grid(alpha=0.25)

        xloc = np.arange(len(feature_cols))
        ax_top.bar(xloc, pca.components_[0])
        ax_top.axhline(0, color="black", lw=0.8)
        ax_top.set_xticks(xloc)
        ax_top.set_xticklabels([])   # niente tick label sotto
        ax_top.set_title("PC1 loadings", fontsize=10)
        ax_top.tick_params(axis='y', labelsize=8)
        ax_top.grid(alpha=0.25)

        # labels sopra il grafico, dritte
        y_top = ax_top.get_ylim()[1]
        for i, feat in enumerate(feature_cols):
            ax_top.text(
                i, y_top * 1.02, feat,
                rotation=0,
                ha="center",
                va="bottom",
                fontsize=7,
                clip_on=False
            )

        ax_top.margins(y=0.20)

        yloc = np.arange(len(feature_cols))
        ax_right.barh(yloc, pca.components_[1])
        ax_right.axvline(0, color="black", lw=0.8)
        ax_right.set_yticks(yloc)
        ax_right.set_yticklabels([])   # niente labels a sinistra
        ax_right.set_title("PC2 loadings", fontsize=10)
        ax_right.tick_params(axis='x', labelsize=8)
        ax_right.grid(alpha=0.25)

        x_right = ax_right.get_xlim()[1]
        for i, feat in enumerate(feature_cols):
            ax_right.text(
                x_right * 1.02, i, feat,
                rotation=0,
                ha="left",
                va="center",
                fontsize=7,
                clip_on=False
            )

        ax_right.margins(x=0.25)

        plt.subplots_adjust(
            left=0.08,
            right=0.88,
            top=0.92,
            bottom=0.08
        )

        plt.show()

    # -----------------------------
    # Tabella finale
    # -----------------------------
    summary_cols = [
        "top_code", "label", "kind", "support", "cluster"
    ] + feature_cols
    summary_cols = [c for c in summary_cols if c in df_vectors.columns]

    df_summary = df_vectors[summary_cols].copy()

    out_analysis = {
        "df_vectors": df_vectors,
        "df_summary": df_summary,
        "feature_cols": feature_cols,
        "X_raw": X,
        "X_proc": X_proc,
        "X_pca": X_pca,
        "pca_model": pca,
        "scaler": scaler,
        "linkage_matrix": Z,
        "cluster_labels": cluster_labels,
        "clustering_space": clustering_space,
        "n_clusters": n_clusters,
        "feature_specs": _normalize_feature_specs(feature_specs),
    }

    return out_analysis

In [ ]:
res = analyze_feature_vectors_from_out(
    out, feature_specs=FEATURE_SPECS, top_codes=TOP_CODES,
    clustering_space="pca", n_clusters=N_CLUSTERS, n_pcs=2,
)


## 4. Project new spectra into the constructed PCA spaces

In [ ]:
def build_injected_feature_vector_table(
    feature_specs,
    inject_spectra,
    inject_titles,
    wavelength_tmp,
    window_spec,
    *,
    xlim=(4000, 400),
    sample_kind="injected",
    sample_support="inj",
):
    """
    Costruisce una tabella feature per i sample nuovi usando:
    - le stesse feature definite in feature_specs
    - le stesse finestre definite in window_spec

    Output:
      df_inj: DataFrame with one sample per raw
      feature_cols: ordered list of feature columns
    """
    specs = _normalize_feature_specs(feature_specs)
    window_spec = normalize_window_spec(window_spec)

    inject_spectra = np.asarray(inject_spectra, dtype=float)
    inject_titles = np.asarray(inject_titles, dtype=str)

    if len(inject_spectra) != len(inject_titles):
        raise ValueError("inject_spectra and inject_titles must have the same length")

    wn_plot, mask, reverse_needed = prepare_common_axis(wavelength_tmp, xlim=xlim)

    rows = []
    feature_cols = [f"{w}__{f}" for w, f in specs]

    for i, title in enumerate(inject_titles):
        y_full = np.asarray(inject_spectra[i], dtype=float)[mask]
        if reverse_needed:
            y_full = y_full[::-1]

        row = {
            "sample_id": f"INJ:{clean_name(title)}:inj",
            "top_code": None,
            "label": clean_name(title),
            "kind": sample_kind,
            "support": sample_support,
            "bottom_code": None,
            "top_ratio": None,
            "stack_type": "injected",
            "n_repeated_layers": None,
            "display_name": clean_name(title),
            "stack_signature": clean_name(title),
            "source": "injected",
        }

        for window_name, feat_name in specs:
            if window_name not in window_spec:
                raise ValueError(f"Window {window_name} not present in window_spec")

            bounds = tuple(window_spec[window_name]["others"])
            lo, hi = min(bounds), max(bounds)
            m = (wn_plot >= lo) & (wn_plot <= hi)

            col_name = f"{window_name}__{feat_name}"

            if np.sum(m) < 3:
                row[col_name] = np.nan
                continue

            feat = compute_window_features(wn_plot[m], y_full[m])

            if feat_name not in feat:
                raise ValueError(f"Feature '{feat_name}' not found in compute_window_features")

            row[col_name] = feat[feat_name]

        rows.append(row)

    df_inj = pd.DataFrame(rows)
    return df_inj, feature_cols

def project_new_spectra_into_existing_pca(
    out, feature_specs, *, inject_spectra, inject_titles,
    wavelength_tmp, window_spec, top_codes=None,
    include_kinds=("layered_raw", "layered_treated", "oil_ref", "pellet_ref"),
    include_single_layer=True, dropna="any", scale_features=True, n_pcs=2,
):
    """Fit PCA on reference spectra only, then project unseen spectra into the same space."""
    df_vectors, feature_cols = build_feature_vector_table_from_out(
        out, feature_specs, top_codes=top_codes,
        include_kinds=include_kinds, include_single_layer=include_single_layer,
        dropna=dropna,
    )
    X_reference = df_vectors[feature_cols].to_numpy(dtype=float)
    if not np.isfinite(X_reference).all():
        raise ValueError("Reference feature vectors contain non-finite values.")
    scaler = StandardScaler() if scale_features else None
    X_reference_scaled = scaler.fit_transform(X_reference) if scaler is not None else X_reference
    pca_model = PCA(n_components=min(n_pcs, *X_reference_scaled.shape))
    reference_scores = pca_model.fit_transform(X_reference_scaled)

    df_new, new_feature_cols = build_injected_feature_vector_table(
        feature_specs=feature_specs, inject_spectra=inject_spectra,
        inject_titles=inject_titles, wavelength_tmp=wavelength_tmp,
        window_spec=window_spec,
    )
    if new_feature_cols != feature_cols:
        raise ValueError("Feature order differs between reference and new spectra.")
    X_new = df_new[feature_cols].to_numpy(dtype=float)
    if not np.isfinite(X_new).all():
        raise ValueError("New feature vectors contain non-finite values.")
    X_new_scaled = scaler.transform(X_new) if scaler is not None else X_new
    new_scores = pca_model.transform(X_new_scaled)

    fig, ax = plt.subplots(figsize=(9, 7))
    for group, indices in df_vectors.groupby("top_code").groups.items():
        idx = df_vectors.index.get_indexer(indices)
        ax.scatter(reference_scores[idx, 0], reference_scores[idx, 1],
                   alpha=0.45, s=28, label=str(group))
    ax.scatter(new_scores[:, 0], new_scores[:, 1], marker="*", s=180,
               c="black", edgecolors="white", linewidths=0.8, label="New samples")
    for title, xy in zip(inject_titles, new_scores):
        ax.annotate(str(title), xy[:2], xytext=(5, 5), textcoords="offset points", fontsize=8)
    ax.set_xlabel(f"PC1 ({pca_model.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca_model.explained_variance_ratio_[1]:.1%})")
    ax.set_title("Projection of new ER-FTIR spectra into reference PCA")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    plt.show()
    return {"pca_model": pca_model, "scaler": scaler, "df_vectors": df_vectors,
            "df_injected": df_new, "X_old_pca": reference_scores,
            "X_new_pca": new_scores, "feature_cols": feature_cols}


In [ ]:
# Add one or more independent SPA files to data/New_Samples/.
new_files = sorted((DATA_ROOT / "New_Samples").glob("*.SPA"))
new_files += sorted((DATA_ROOT / "New_Samples").glob("*.spa"))
new_files = list(dict.fromkeys(new_files))
if new_files:
    new_raw, new_titles, new_axis = load_group(DATA_ROOT / "New_Samples")
    if len(new_axis) != len(wavelength_tmp) or not np.allclose(new_axis, wavelength_tmp):
        raise ValueError("New spectra must use the reference wavenumber grid.")
    new_derivative = preprocess_group(new_raw, wavelength_tmp)
    projection_result = project_new_spectra_into_existing_pca(
        out, FEATURE_SPECS,
        inject_spectra=new_derivative, inject_titles=new_titles,
        wavelength_tmp=wavelength_tmp, window_spec=WINDOW_SPEC,
        top_codes=TOP_CODES,
    )
    print(pd.DataFrame({
        "sample": new_titles,
        "PC1": projection_result["X_new_pca"][:, 0],
        "PC2": projection_result["X_new_pca"][:, 1],
    }).to_string(index=False))
else:
    print("Add SPA files to data/New_Samples/ to run the PCA projection example.")


## 4. KDE with leave-one-out bandwidth optimization

In [ ]:
def kde_hdr_threshold(Z, alpha=0.90):

    z = np.asarray(Z, dtype=float).ravel()

    order = np.argsort(z)[::-1]
    z_sorted = z[order]

    cumulative = np.cumsum(z_sorted)
    cumulative /= cumulative[-1]

    idx = np.searchsorted(
        cumulative,
        alpha,
        side="left"
    )

    idx = min(
        idx,
        len(z_sorted) - 1
    )

    return float(
        z_sorted[idx]
    )

def get_topcode_pca_points(
    res,
    top_code="VC",
    cluster_id=None
):

    df = res[
        "df_vectors"
    ].copy()

    X_pca = np.asarray(
        res[
            "X_pca"
        ]
    )

    cluster_labels = np.asarray(
        res[
            "cluster_labels"
        ]
    )


    df["PC1"] = X_pca[:, 0]
    df["PC2"] = X_pca[:, 1]
    df["cluster_hca"] = (
        cluster_labels
    )


    df_top = df[
        df[
            "top_code"
        ]
        .astype(
            str
        )
        .eq(
            str(
                top_code
            )
        )
    ].copy()


    if len(
        df_top
    ) < 4:

        raise ValueError(
            f"Too few points for "
            f"{top_code}: "
            f"n = {len(df_top)}"
        )


    if cluster_id is None:

        cluster_id = (
            df_top[
                "cluster_hca"
            ]
            .value_counts()
            .idxmax()
        )


    df_cluster = df_top[
        df_top[
            "cluster_hca"
        ]
        ==
        cluster_id
    ].copy()


    if len(
        df_cluster
    ) < 4:

        raise ValueError(
            f"Too few points for "
            f"{top_code} in cluster "
            f"{cluster_id}: "
            f"n = {len(df_cluster)}"
        )


    points_all = df_top[
        [
            "PC1",
            "PC2"
        ]
    ].to_numpy(
        dtype=float
    )


    points_cluster = df_cluster[
        [
            "PC1",
            "PC2"
        ]
    ].to_numpy(
        dtype=float
    )


    return {

        "df_all":
            df_top,

        "df_cluster":
            df_cluster,

        "points_all":
            points_all,

        "points_cluster":
            points_cluster,

        "cluster_id":
            cluster_id
    }

def loo_kde_loglikelihood(
    points,
    bw_scale,
    density_floor=1e-300
):

    points = np.asarray(
        points,
        dtype=float
    )


    n = len(
        points
    )


    log_likelihoods = np.empty(
        n
    )


    for i in range(
        n
    ):

        train_points = np.delete(
            points,
            i,
            axis=0
        )


        test_point = points[
            i
        ]


        kde = gaussian_kde(
            train_points.T,
            bw_method=lambda obj:
                obj.scotts_factor()
                *
                bw_scale
        )


        density = float(
            kde(
                test_point.reshape(
                    2,
                    1
                )
            )[0]
        )


        density = max(
            density,
            density_floor
        )


        log_likelihoods[
            i
        ] = np.log(
            density
        )


    return {

        "bw_scale":
            bw_scale,

        "log_likelihoods":
            log_likelihoods,

        "mean_log_likelihood":
            np.mean(
                log_likelihoods
            ),

        "sum_log_likelihood":
            np.sum(
                log_likelihoods
            ),

        "std_log_likelihood":
            np.std(
                log_likelihoods,
                ddof=1
            )
    }

def scan_loo_kde_bandwidth(
    points,
    bw_scales
):

    rows = []

    pointwise_results = {}


    for scale in bw_scales:

        result = (
            loo_kde_loglikelihood(
                points,
                bw_scale=scale
            )
        )


        rows.append({

            "bw_scale":
                scale,

            "mean_log_likelihood":
                result[
                    "mean_log_likelihood"
                ],

            "sum_log_likelihood":
                result[
                    "sum_log_likelihood"
                ],

            "std_log_likelihood":
                result[
                    "std_log_likelihood"
                ]
        })


        pointwise_results[
            scale
        ] = result[
            "log_likelihoods"
        ]


    df_scan = pd.DataFrame(
        rows
    )


    best_idx = (
        df_scan[
            "mean_log_likelihood"
        ]
        .idxmax()
    )


    best_row = df_scan.loc[
        best_idx
    ].copy()


    return {

        "scan":
            df_scan,

        "best_scale":
            float(
                best_row[
                    "bw_scale"
                ]
            ),

        "best_mean_loglik":
            float(
                best_row[
                    "mean_log_likelihood"
                ]
            ),

        "best_sum_loglik":
            float(
                best_row[
                    "sum_log_likelihood"
                ]
            ),

        "pointwise":
            pointwise_results
    }

def compare_loo_bandwidth_all_vs_hca(
    res,
    top_code="VC",
    cluster_id=None,
    bw_min=0.40,
    bw_max=2.00,
    n_bw=161,
    figsize=(13, 5)
):

    data = get_topcode_pca_points(
        res,
        top_code=top_code,
        cluster_id=cluster_id
    )


    points_all = data[
        "points_all"
    ]

    points_cluster = data[
        "points_cluster"
    ]

    cluster_id = data[
        "cluster_id"
    ]


    bw_scales = np.linspace(
        bw_min,
        bw_max,
        n_bw
    )


    result_all = (
        scan_loo_kde_bandwidth(
            points_all,
            bw_scales
        )
    )


    result_cluster = (
        scan_loo_kde_bandwidth(
            points_cluster,
            bw_scales
        )
    )


    best_all = result_all[
        "best_scale"
    ]

    best_cluster = (
        result_cluster[
            "best_scale"
        ]
    )


    print(
        "=" * 70
    )

    print(
        f"{top_code} — Likelihood "
        "Cross-Validation KDE"
    )

    print(
        "=" * 70
    )

    print()

    print(
        f"All {top_code}: "
        f"n = {len(points_all)}"
    )


    print(
        f"{top_code} in cluster HCA "
        f"{cluster_id}: "
        f"n = {len(points_cluster)}"
    )


    print()

    print(
        "-" * 70
    )


    print(
        "OPTIMAL — all points"
    )


    print(
        f"Scott multiplier = "
        f"{best_all:.3f}"
    )


    print(
        f"Mean LOO log-likelihood = "
        f"{result_all['best_mean_loglik']:.6f}"
    )


    print()

    print(
        "OPTIMAL — HCA-selected"
    )


    print(
        f"Scott multiplier = "
        f"{best_cluster:.3f}"
    )


    print(
        f"Mean LOO log-likelihood = "
        f"{result_cluster['best_mean_loglik']:.6f}"
    )


    print(
        "-" * 70
    )


    fig, ax = plt.subplots(
        figsize=figsize
    )


    ax.plot(
        result_all[
            "scan"
        ][
            "bw_scale"
        ],
        result_all[
            "scan"
        ][
            "mean_log_likelihood"
        ],
        linewidth=2.3,
        label=(
            f"All {top_code} "
            f"(n={len(points_all)})"
        )
    )


    ax.plot(
        result_cluster[
            "scan"
        ][
            "bw_scale"
        ],
        result_cluster[
            "scan"
        ][
            "mean_log_likelihood"
        ],
        linewidth=2.3,
        linestyle="--",
        label=(
            f"{top_code} in HCA cluster "
            f"{cluster_id} "
            f"(n={len(points_cluster)})"
        )
    )


    ax.axvline(
        best_all,
        linestyle=":",
        linewidth=1.7
    )


    ax.axvline(
        best_cluster,
        linestyle=":",
        linewidth=1.7
    )


    ax.scatter(
        best_all,
        result_all[
            "best_mean_loglik"
        ],
        s=90,
        zorder=5
    )


    ax.scatter(
        best_cluster,
        result_cluster[
            "best_mean_loglik"
        ],
        s=90,
        zorder=5
    )


    ax.annotate(
        f"All: {best_all:.2f}",
        (
            best_all,
            result_all[
                "best_mean_loglik"
            ]
        ),
        xytext=(
            8,
            10
        ),
        textcoords=(
            "offset points"
        )
    )


    ax.annotate(
        f"HCA: {best_cluster:.2f}",
        (
            best_cluster,
            result_cluster[
                "best_mean_loglik"
            ]
        ),
        xytext=(
            8,
            -18
        ),
        textcoords=(
            "offset points"
        )
    )


    ax.axvline(
        1.0,
        color="0.5",
        linestyle="--",
        linewidth=1.2,
        label="Scott standard"
    )


    ax.set_xlabel(
        "Bandwidth multiplier relative to Scott"
    )


    ax.set_ylabel(
        "Mean leave-one-out log-likelihood"
    )


    ax.set_title(
        f"{top_code} — KDE bandwidth selection "
        "by LOO likelihood CV"
    )


    ax.grid(
        alpha=0.20
    )


    ax.legend(
        frameon=False
    )


    plt.tight_layout()
    plt.show()


    return {

        "top_code":
            top_code,

        "cluster_id":
            cluster_id,

        "points_all":
            points_all,

        "points_cluster":
            points_cluster,

        "result_all":
            result_all,

        "result_cluster":
            result_cluster,

        "best_scale_all":
            best_all,

        "best_scale_cluster":
            best_cluster,

        "df_all":
            data[
                "df_all"
            ],

        "df_cluster":
            data[
                "df_cluster"
            ]
    }

def plot_best_vc_kde_heatmaps(
    res,
    lcv_vc,
    top_code="VC",
    use_hca=False,
    alpha=0.90,
    gridsize=350,
    pad_fraction=0.40,
    cmap="cividis",
    elev=28,
    azim=-58,
    figsize=(20, 6.5)
):

    # ========================================================
    # 1. DATI
    # ========================================================

    df = res["df_vectors"].copy()

    X_pca = np.asarray(
        res["X_pca"]
    )

    clusters = np.asarray(
        res["cluster_labels"]
    )

    df["PC1"] = X_pca[:, 0]
    df["PC2"] = X_pca[:, 1]
    df["cluster_hca"] = clusters


    df_top_all = df[
        df["top_code"].eq(top_code)
    ].copy()


    points_all = df_top_all[
        ["PC1", "PC2"]
    ].to_numpy(
        dtype=float
    )


    # ========================================================
    # 2. SCELTA:
    #    ALL oppure HCA-selected
    # ========================================================

    if use_hca:

        cluster_id = int(
            lcv_vc["cluster_id"]
        )

        df_fit = df_top_all[
            df_top_all["cluster_hca"]
            ==
            cluster_id
        ].copy()

        points = df_fit[
            ["PC1", "PC2"]
        ].to_numpy(
            dtype=float
        )

        bw_scale = float(
            lcv_vc[
                "best_scale_cluster"
            ]
        )

        fit_label = (
            f"HCA-selected {top_code} "
            f"(cluster {cluster_id})"
        )

    else:

        cluster_id = None

        points = points_all.copy()

        bw_scale = float(
            lcv_vc[
                "best_scale_all"
            ]
        )

        fit_label = (
            f"All {top_code}"
        )


    # ========================================================
    # 3. KDE
    # ========================================================

    kde = gaussian_kde(
        points.T,
        bw_method=lambda obj:
            obj.scotts_factor()
            *
            bw_scale
    )


    xmin = points_all[:, 0].min()
    xmax = points_all[:, 0].max()

    ymin = points_all[:, 1].min()
    ymax = points_all[:, 1].max()


    dx = xmax - xmin
    dy = ymax - ymin


    if dx == 0:
        dx = 1.0

    if dy == 0:
        dy = 1.0


    xmin -= (
        pad_fraction
        *
        dx
    )

    xmax += (
        pad_fraction
        *
        dx
    )

    ymin -= (
        pad_fraction
        *
        dy
    )

    ymax += (
        pad_fraction
        *
        dy
    )


    x_grid = np.linspace(
        xmin,
        xmax,
        gridsize
    )

    y_grid = np.linspace(
        ymin,
        ymax,
        gridsize
    )


    XX, YY = np.meshgrid(
        x_grid,
        y_grid
    )


    grid = np.vstack([
        XX.ravel(),
        YY.ravel()
    ])


    # ========================================================
    # 5. KDE Density
    # ========================================================

    Z = kde(
        grid
    ).reshape(
        XX.shape
    )


    hdr_threshold = (
        kde_hdr_threshold(
            Z,
            alpha=alpha
        )
    )

    norm = Normalize(
        vmin=0,
        vmax=Z.max()
    )


    levels = np.linspace(
        0,
        Z.max(),
        35
    )

    fig = plt.figure(
        figsize=figsize
    )

    ax1 = fig.add_subplot(
        1,
        3,
        1
    )


    ax1.contourf(
        XX,
        YY,
        Z,
        levels=levels,
        cmap=cmap,
        norm=norm
    )


    ax1.contour(
        XX,
        YY,
        Z,
        levels=10,
        colors="black",
        linewidths=0.65,
        alpha=0.55
    )


    ax1.contour(
        XX,
        YY,
        Z,
        levels=[
            hdr_threshold
        ],
        colors="black",
        linewidths=2.2
    )


    ax1.set_xlabel(
        "PC1"
    )

    ax1.set_ylabel(
        "PC2"
    )


    ax1.set_title(
        f"{fit_label} — KDE density\n"
        f"LOO: Scott × {bw_scale:.3f}"
    )


    ax1.set_xlim(
        xmin,
        xmax
    )

    ax1.set_ylim(
        ymin,
        ymax
    )

    ax1.grid(
        alpha=0.12
    )

    ax2 = fig.add_subplot(
        1,
        3,
        2
    )


    ax2.contourf(
        XX,
        YY,
        Z,
        levels=levels,
        cmap=cmap,
        norm=norm
    )


    ax2.contour(
        XX,
        YY,
        Z,
        levels=10,
        colors="black",
        linewidths=0.65,
        alpha=0.55
    )


    ax2.contour(
        XX,
        YY,
        Z,
        levels=[
            hdr_threshold
        ],
        colors="black",
        linewidths=2.2
    )


    if use_hca:

        ax2.scatter(
            points_all[:, 0],
            points_all[:, 1],
            s=65,
            facecolors="none",
            edgecolors="0.55",
            linewidths=1.3,
            zorder=8,
            label=f"All {top_code}"
        )

    ax2.scatter(
        points[:, 0],
        points[:, 1],
        s=60,
        facecolor="white",
        edgecolor="black",
        linewidth=0.9,
        alpha=0.95,
        zorder=10,
        label="Used for KDE"
    )


    ax2.set_xlabel(
        "PC1"
    )

    ax2.set_ylabel(
        "PC2"
    )


    ax2.set_title(
        f"{fit_label} — KDE + samples\n"
        f"n={len(points)} | "
        f"{alpha:.0%} HDR"
    )


    ax2.set_xlim(
        xmin,
        xmax
    )

    ax2.set_ylim(
        ymin,
        ymax
    )


    ax2.grid(
        alpha=0.12
    )

    ax2.legend(
        frameon=False
    )


    # ========================================================
    # C) 3D SURFACE PLOT
    # ========================================================

    ax3 = fig.add_subplot(
        1,
        3,
        3,
        projection="3d"
    )


    ax3.plot_surface(
        XX,
        YY,
        Z,
        cmap=cmap,
        norm=norm,
        rstride=2,
        cstride=2,
        linewidth=0,
        antialiased=True,
        alpha=0.92
    )


    ax3.contourf(
        XX,
        YY,
        Z,
        zdir="z",
        offset=0,
        levels=levels,
        cmap=cmap,
        norm=norm,
        alpha=0.90
    )


    ax3.contour(
        XX,
        YY,
        Z,
        zdir="z",
        offset=0,
        levels=10,
        colors="black",
        linewidths=0.55,
        alpha=0.60
    )


    ax3.contour(
        XX,
        YY,
        Z,
        zdir="z",
        offset=0,
        levels=[
            hdr_threshold
        ],
        colors="black",
        linewidths=2.0
    )


    ax3.set_xlabel(
        "PC1"
    )

    ax3.set_ylabel(
        "PC2"
    )

    ax3.set_zlabel(
        rf"$f(PC1,PC2\mid {top_code})$"
    )


    ax3.set_title(
        f"{fit_label} — KDE surface\n"
        f"LOO: Scott × {bw_scale:.3f}"
    )


    ax3.set_xlim(
        xmin,
        xmax
    )

    ax3.set_ylim(
        ymin,
        ymax
    )

    ax3.set_zlim(
        0,
        Z.max() * 1.05
    )


    ax3.view_init(
        elev=elev,
        azim=azim
    )

    sm = ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    sm.set_array([])


    cbar = fig.colorbar(
        sm,
        ax=[
            ax1,
            ax2,
            ax3
        ],
        shrink=0.77,
        pad=0.035
    )


    cbar.set_label(
        rf"KDE density  $f(PC1,PC2\mid {top_code})$"
    )


    fig.suptitle(
        f"{top_code} — LOO-optimized KDE — "
        f"{'HCA-selected' if use_hca else 'All GT'}",
        fontsize=15
    )


    plt.show()


    return {

        "kde":
            kde,

        "points":
            points,

        "points_all":
            points_all,

        "use_hca":
            use_hca,

        "cluster_id":
            cluster_id,

        "bw_scale":
            bw_scale,

        "kde_factor":
            kde.factor,

        "XX":
            XX,

        "YY":
            YY,

        "Z":
            Z,

        "hdr_threshold":
            hdr_threshold
    }

In [ ]:
# None selects the main HCA cluster for TARGET_CODE
TARGET_CLUSTER = None

loo_result = compare_loo_bandwidth_all_vs_hca(
    res, top_code=TARGET_CODE, cluster_id=TARGET_CLUSTER,
    bw_min=BW_MIN, bw_max=BW_MAX, n_bw=N_BW,
)
print("Best Scott multiplier (all):", loo_result["best_scale_all"])
print("Best Scott multiplier (HCA):", loo_result["best_scale_cluster"])


In [ ]:
# Plot the LOO-optimized 2D KDE in PCA score space.
kde_result = plot_best_vc_kde_heatmaps(
    res, loo_result, use_hca=True, top_code=TARGET_CODE,
    alpha=0.90, gridsize=250,
)
